# V9 Phase 3: Fusion & Training

## Goal
Fuse FastText text embeddings with ResNet-50 visual embeddings and train the final alignment model.

## Process
1. Load V7 FastText model (768d text embeddings)
2. Load V9 visual embeddings (768d per Gardiner code)
3. Create Transliteration → Gardiner Code mapping (using V6 lexicon)
4. Fuse: Text (768d) + Visual (768d) = 1536d
5. Train Ridge Regression alignment (1536d → 300d English GloVe)
6. Evaluate on test set and compare to V7 baseline (29.10%)

In [1]:
import logging
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from gensim.models import FastText, KeyedVectors
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Setup paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPO_ROOT = PROJECT_ROOT.parent

print(f'Project Root: {PROJECT_ROOT}')
print(f'Repository Root: {REPO_ROOT}')

Project Root: /Users/crashy/Development/heiroglyphy/heiro_v9_use_visuals_again
Repository Root: /Users/crashy/Development/heiroglyphy


## 1. Load FastText Text Embeddings (V7)

In [2]:
# Load V7 FastText model
v7_model_path = REPO_ROOT / 'heiro_v7_FastTextVisual/models/fasttext_v7.model'
print(f'Loading V7 FastText model from {v7_model_path}...')

fasttext_model = FastText.load(str(v7_model_path))
text_embeddings = fasttext_model.wv

print(f'✓ Loaded {len(text_embeddings)} text embeddings (768d)')

Loading V7 FastText model from /Users/crashy/Development/heiroglyphy/heiro_v7_FastTextVisual/models/fasttext_v7.model...
✓ Loaded 80662 text embeddings (768d)


## 2. Load Visual Embeddings (V9)

In [3]:
# Load visual embeddings
visual_path = PROJECT_ROOT / 'data/processed/visual_embeddings_768d.pkl'
print(f'Loading visual embeddings from {visual_path}...')

with open(visual_path, 'rb') as f:
    visual_embeddings = pickle.load(f)

print(f'✓ Loaded {len(visual_embeddings)} visual embeddings (768d per Gardiner code)')

Loading visual embeddings from /Users/crashy/Development/heiroglyphy/heiro_v9_use_visuals_again/data/processed/visual_embeddings_768d.pkl...
✓ Loaded 115 visual embeddings (768d per Gardiner code)


## 3. Create Transliteration → Gardiner Code Mapping

In [4]:
# Load V6 hieroglyph lexicon
lexicon_path = REPO_ROOT / 'heiro_v6_BERT/data/processed/hieroglyph_lexicon.csv'
print(f'Loading lexicon from {lexicon_path}...')

lexicon_df = pd.read_csv(lexicon_path)
print(f'✓ Loaded lexicon with {len(lexicon_df)} entries')
print(f'\nColumns: {list(lexicon_df.columns)}')
print(f'\nSample entries:')
print(lexicon_df.head())

Loading lexicon from /Users/crashy/Development/heiroglyphy/heiro_v6_BERT/data/processed/hieroglyph_lexicon.csv...
✓ Loaded lexicon with 1071 entries

Columns: ['unicode', 'character', 'glyph_name', 'gardiner_code']

Sample entries:
   unicode character glyph_name gardiner_code
0  U+13000         𓀀         a1       U+13000
1  U+13001         𓀁         a2       U+13001
2  U+13002         𓀂         a3       U+13002
3  U+13003         𓀃         a4       U+13003
4  U+13004         𓀄         a5       U+13004


In [5]:
# Create mapping: Gardiner code → Unicode/Character
# The lexicon has: unicode, character, glyph_name, gardiner_code
# We need to map Gardiner codes to something we can match with FastText vocab

# For now, let's create a simple mapping
# Gardiner code (e.g., 'D21') → lowercase glyph name (e.g., 'd21')
gardiner_to_glyph = dict(zip(
    lexicon_df['gardiner_code'].str.replace('U+', ''),  # Remove U+ prefix if present
    lexicon_df['glyph_name'].str.lower()
))

print(f'Created mapping for {len(gardiner_to_glyph)} Gardiner codes')
print(f'\nSample mappings:')
for i, (gardiner, glyph) in enumerate(list(gardiner_to_glyph.items())[:10]):
    print(f'  {gardiner} → {glyph}')

Created mapping for 1071 Gardiner codes

Sample mappings:
  13000 → a1
  13001 → a2
  13002 → a3
  13003 → a4
  13004 → a5
  13005 → a5a
  13006 → a6
  13007 → a6a
  13008 → a6b
  13009 → a7


## 4. Fuse Text + Visual Embeddings

For each word in the FastText vocabulary:
1. Get text embedding (768d)
2. Try to find corresponding Gardiner code(s)
3. If found, get visual embedding (768d)
4. Concatenate: [text, visual] = 1536d
5. If no visual found, use zeros: [text, zeros] = 1536d

In [6]:
# Create fused embeddings
fused_embeddings = {}
visual_match_count = 0
visual_miss_count = 0

print('Fusing text + visual embeddings...')
for word in tqdm(text_embeddings.index_to_key, desc='Processing vocabulary'):
    # Get text embedding
    text_vec = text_embeddings[word]
    
    # Try to find visual embedding
    # Strategy: Check if word matches any Gardiner code pattern
    visual_vec = None
    
    # Check if word is a Gardiner code (e.g., 'd21', 'f35')
    word_upper = word.upper()
    if word_upper in visual_embeddings:
        visual_vec = visual_embeddings[word_upper]
        visual_match_count += 1
    else:
        # No visual match, use zeros
        visual_vec = np.zeros(768)
        visual_miss_count += 1
    
    # Fuse: concatenate text + visual
    fused_vec = np.concatenate([text_vec, visual_vec])
    fused_embeddings[word] = fused_vec

print(f'\n✓ Created {len(fused_embeddings)} fused embeddings (1536d)')
print(f'  Visual matches: {visual_match_count} ({visual_match_count/len(fused_embeddings)*100:.1f}%)')
print(f'  Visual misses: {visual_miss_count} ({visual_miss_count/len(fused_embeddings)*100:.1f}%)')

Fusing text + visual embeddings...


Processing vocabulary:   0%|          | 0/80662 [00:00<?, ?it/s]


✓ Created 80662 fused embeddings (1536d)
  Visual matches: 0 (0.0%)
  Visual misses: 80662 (100.0%)


## 5. Load Anchors and English Embeddings

In [7]:
# Load V7 anchors (revert to V7 baseline, not V8)
anchors_path = REPO_ROOT / 'heiro_v6_BERT/data/processed/anchors.json'
print(f'Loading anchors from {anchors_path}...')

with open(anchors_path, 'r', encoding='utf-8') as f:
    anchors = json.load(f)

print(f'✓ Loaded {len(anchors)} anchor pairs')

Loading anchors from /Users/crashy/Development/heiroglyphy/heiro_v6_BERT/data/processed/anchors.json...
✓ Loaded 8541 anchor pairs


In [8]:
# Load GloVe English embeddings
glove_path = REPO_ROOT / 'heiro_v5_getdata/data/processed/glove.6B.300d.txt'
print(f'Loading GloVe from {glove_path}...')
print('(This may take a minute...)')

english_embeddings = KeyedVectors.load_word2vec_format(str(glove_path), binary=False, no_header=True)

print(f'✓ Loaded {len(english_embeddings)} English embeddings (300d)')

Loading GloVe from /Users/crashy/Development/heiroglyphy/heiro_v5_getdata/data/processed/glove.6B.300d.txt...
(This may take a minute...)
✓ Loaded 400000 English embeddings (300d)


## 6. Prepare Training Data

In [9]:
# Prepare X (Egyptian fused) and Y (English) matrices
X = []
Y = []
valid_anchors = []

for anchor in anchors:
    egy_word = anchor['hieroglyphic']
    eng_word = anchor['english'].lower()
    
    if egy_word in fused_embeddings and eng_word in english_embeddings:
        X.append(fused_embeddings[egy_word])
        Y.append(english_embeddings[eng_word])
        valid_anchors.append((egy_word, eng_word))

X = np.array(X)
Y = np.array(Y)

print(f'Valid anchors: {len(X)} / {len(anchors)} ({len(X)/len(anchors)*100:.1f}%)')
print(f'X shape: {X.shape} (Egyptian fused embeddings)')
print(f'Y shape: {Y.shape} (English embeddings)')

Valid anchors: 6700 / 8541 (78.4%)
X shape: (6700, 1536) (Egyptian fused embeddings)
Y shape: (6700, 300) (English embeddings)


## 7. Train Alignment Model

In [10]:
# Split data
X_train, X_test, Y_train, Y_test, anchors_train, anchors_test = train_test_split(
    X, Y, valid_anchors, test_size=0.2, random_state=42
)

print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

# Train Ridge Regression
print('\nTraining Ridge Regression alignment...')
aligner = Ridge(alpha=1.0)
aligner.fit(X_train, Y_train)

print(f'R² Score on Train: {aligner.score(X_train, Y_train):.4f}')
print(f'R² Score on Test: {aligner.score(X_test, Y_test):.4f}')

Train size: 5360, Test size: 1340

Training Ridge Regression alignment...
R² Score on Train: 0.0975
R² Score on Test: 0.0408


## 8. Evaluate Accuracy

In [11]:
# Evaluate on test set
print('Evaluating on test set...')
correct_top1 = 0
correct_top5 = 0
correct_top10 = 0
total = len(X_test)

Y_pred = aligner.predict(X_test)

for i in tqdm(range(total), desc='Evaluating'):
    pred_vec = Y_pred[i]
    true_word = anchors_test[i][1]
    
    # Find nearest neighbors in English space
    neighbors = english_embeddings.similar_by_vector(pred_vec, topn=10)
    neighbor_words = [w for w, s in neighbors]
    
    if true_word == neighbor_words[0]:
        correct_top1 += 1
    if true_word in neighbor_words[:5]:
        correct_top5 += 1
    if true_word in neighbor_words[:10]:
        correct_top10 += 1

acc_top1 = correct_top1 / total * 100
acc_top5 = correct_top5 / total * 100
acc_top10 = correct_top10 / total * 100

print(f'\n=========================================')
print(f'V9 Results (Test Set N={total})')
print(f'=========================================')
print(f'Top-1 Accuracy:  {acc_top1:.2f}%')
print(f'Top-5 Accuracy:  {acc_top5:.2f}%')
print(f'Top-10 Accuracy: {acc_top10:.2f}%')
print(f'=========================================')
print(f'\nComparison to V7 Baseline:')
print(f'  V7 (Text-Only):  29.10%')
print(f'  V9 (Text+Visual): {acc_top1:.2f}%')
print(f'  Delta:           {acc_top1 - 29.10:+.2f}%')

Evaluating on test set...


Evaluating:   0%|          | 0/1340 [00:00<?, ?it/s]


V9 Results (Test Set N=1340)
Top-1 Accuracy:  30.52%
Top-5 Accuracy:  37.54%
Top-10 Accuracy: 41.79%

Comparison to V7 Baseline:
  V7 (Text-Only):  29.10%
  V9 (Text+Visual): 30.52%
  Delta:           +1.42%


## 9. Save Results

## 10. Cleanup Memory

Free up RAM by deleting large objects we no longer need.

In [13]:
import gc

# Delete large objects
print('Freeing up memory...')

# Delete models and embeddings
del fasttext_model
del text_embeddings
del visual_embeddings
del english_embeddings
del fused_embeddings

# Delete training data
del X, Y, X_train, X_test, Y_train, Y_test, Y_pred
del anchors, valid_anchors, anchors_train, anchors_test

# Run garbage collector
gc.collect()

print('✓ Memory freed!')
print('\nYou can now safely close this notebook or continue with other work.')

Freeing up memory...
✓ Memory freed!

You can now safely close this notebook or continue with other work.


In [12]:
# Save results
results = {
    'total_anchors': len(anchors),
    'valid_anchors': len(valid_anchors),
    'visual_match_rate': visual_match_count / len(fused_embeddings),
    'top1_accuracy': acc_top1,
    'top5_accuracy': acc_top5,
    'top10_accuracy': acc_top10,
    'v7_baseline': 29.10,
    'improvement': acc_top1 - 29.10,
    'status': 'success'
}

output_path = PROJECT_ROOT / 'results.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'\n✓ Results saved to {output_path}')
print(json.dumps(results, indent=2))


✓ Results saved to /Users/crashy/Development/heiroglyphy/heiro_v9_use_visuals_again/results.json
{
  "total_anchors": 8541,
  "valid_anchors": 6700,
  "visual_match_rate": 0.0,
  "top1_accuracy": 30.522388059701495,
  "top5_accuracy": 37.53731343283582,
  "top10_accuracy": 41.7910447761194,
  "v7_baseline": 29.1,
  "improvement": 1.4223880597014933,
  "status": "success"
}
